# AOBA Climate Model — Google Colab (A100 / H100) Automatic CUDA Spin-up

**AOBA** は実行環境のハードウェアを自動検知し、Apple Silicon (`mps`) では Metal シェーダー、NVIDIA GPU (`cuda`) では全43個の融合 CUDA C++17 カーネル（`src/aoba/kernels_cuda.py`）へ自動切替して高速積分を行います。

### おすすめランタイム設定（Google Colab）
1. メニュー **「ランタイム」→「ランタイムのタイプを変更」** で **A100 GPU**（CU効率最強）または **H100 GPU**（最速）を選択してください。
   - *注意*: L4 (300 GB/s) は M1 Max (400 GB/s) よりメモリ帯域が狭いため、A100/H100 の利用を推奨します。
2. 毎月末に `restart_mid.pt` が自動保存されます。出力先を **Google Drive** に配置しておけば、Colab の 24時間上限で終了しても次回セルを再実行するだけで無欠損レジュームします。

In [ ]:
# 1. Check detected NVIDIA GPU & CUDA toolchain
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'CUDA GPU runtime not selected! Switch Runtime -> Change runtime type -> A100/H100 GPU.'
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
cc = torch.cuda.get_device_capability(0)
print(f'Detected GPU: {name} | VRAM: {vram:.1f} GB | Compute Capability: sm_{cc[0]}{cc[1]} | PyTorch {torch.__version__} (CUDA {torch.version.cuda})')
if any(k in name for k in ('T4', 'L4')):
    print('⚠️ Note: T4/L4 has lower memory bandwidth than Apple M1 Max (400 GB/s). Switch to A100 or H100 for 3-8x speedup!')

In [ ]:
# 2. Mount Google Drive for persistent checkpoints, Zarr outputs, Earth2014 data, and CUDA build cache
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/aoba_colab')
RUN_NAME = 'glevel5_picontrol_30yr'  # Change run directory name if starting a separate experiment
OUT_DIR = DRIVE_BASE / 'outputs' / RUN_NAME
DATA_DIR = DRIVE_BASE / 'data'
CUDA_CACHE_DIR = DRIVE_BASE / 'cuda_cache'

for d in (OUT_DIR, DATA_DIR, CUDA_CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

os.environ['AOBA_DATA_DIR'] = str(DATA_DIR)
os.environ['AOBA_CUDA_CACHE_DIR'] = str(CUDA_CACHE_DIR)
print('Output dir    :', OUT_DIR)
print('Data cache dir:', DATA_DIR)
print('CUDA cache dir:', CUDA_CACHE_DIR)

In [ ]:
# 3. Clone / update AOBA repository & install lightweight I/O dependency (zarr + xarray)
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/aoba')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/k1bb/aoba.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

!pip install -q zarr xarray ninja
!pip install -q --no-build-isolation --no-deps -e /content/aoba

In [ ]:
# 4. Compile & verify all 43 fused CUDA kernels (cached on Google Drive after first ~25s build)
import os, sys, time
sys.path.insert(0, '/content/aoba/src')

from aoba.runtime import detect_hardware_profile, apply_runtime_tuning, ensure_earth2014_data
from aoba.kernels import get_active_gpu_backend

hw = apply_runtime_tuning('cuda')
t0 = time.perf_counter()
backend = get_active_gpu_backend('cuda')
print(f'Hardware: {hw}')
print(f'Active fused GPU kernel backend: {backend} (ready in {time.perf_counter() - t0:.1f} s)')
assert backend == 'cuda', 'Expected native CUDA fused kernels!'

# Auto-download Earth2014 5-arcmin topography/bathymetry grids (once, cached in Google Drive)
ensure_earth2014_data()

# Run native CUDA vs CPU numerical parity test
!PYTHONPATH=/content/aoba/src pytest /content/aoba/tests/test_cuda_kernels.py -k test_real_cuda_backend_when_available -v

In [ ]:
# 5. Optional: Measure live G5 8x1 44-layer throughput on this A100/H100 (10 steps = 2 sim-hours)
import time, torch
from aoba.coupled_model import AOBAConfig, init_aoba_model, step_aoba_model
from aoba.runtime import synchronize_device

cfg_bench = AOBAConfig()  # Default: G5 8x1 44L piControl + CKD + 3D OGCM + 2M microphysics
m = init_aoba_model(cfg_bench, device='cuda')
# Warmup 2 steps
for _ in range(2):
    step_aoba_model(m)
synchronize_device('cuda')

N_STEPS = 10
t0 = time.perf_counter()
for _ in range(N_STEPS):
    d = step_aoba_model(m)
synchronize_device('cuda')
dt_ms = (time.perf_counter() - t0) * 1000.0 / N_STEPS
s_per_day = dt_ms * (86400.0 / cfg_bench.dt_host) / 1000.0
yr_per_24h = 86400.0 / (s_per_day * 365.0)
print(f'G5 8x1 44L (All 3 Pillars ON): {dt_ms:.1f} ms/step | {s_per_day:.2f} s/sim_day | ~{yr_per_24h:.1f} sim-years / 24h')
del m
torch.cuda.empty_cache()

In [ ]:
# 6. Launch / auto-resume multi-year piControl spin-up (monthly checkpoints saved to Google Drive)
# Re-running this cell after a 24h Colab session limit automatically resumes from the latest restart_mid.pt / restart.pt!
YEARS = 30

!PYTHONPATH=/content/aoba/src python -u /content/aoba/scripts/run_g5_30yr.py \
    --years {YEARS} \
    --out-dir "{OUT_DIR}" \
    --device cuda